# JRA-3Q 海面更正気圧（気圧配置・日本域）一括ダウンロード（Colab版）

手元のPC/ネットワークから `github.com` や GDEX（データ配布元）へのHTTPS通信がブロックされる環境向けに、Google Colab上でダウンロードするノートブックです。Colab（Googleのサーバー）から直接ダウンロードするので、手元の回線・セキュリティソフトの制限は関係なくなります。

## 方式：全球ファイルをダウンロード → Colab上で日本域に切り出し
GDEXの**静的ファイルサーバー**から月ごとの全球ファイル（約84MB）を普通にダウンロードし、Colab上で日本周辺（既定: 北緯15〜50度、東経115〜155度）だけを切り出して約4.8MBで保存、元の全球ファイルはすぐ削除します。

- **通信量**は全185ヶ月で約15GB（Colab⇔GDEX間なので手元の回線は使いません）
- **最終的にPCへ持ち帰るのは1GB弱**（切り出し済みのみ）
- Colabのディスクも圧迫しません（全球ファイルは1件ずつ使い捨て）

### なぜ「一見無駄な」全球ダウンロードなのか
サーバー側で切り出してもらう賢い方法を2つ試しましたが、どちらもこのサーバーでは使えませんでした:
- **NCSS**（サーバー側切り出し）… 大半の月がタイムアウト。成功率1割以下
- **OPeNDAP**（必要な範囲だけ読む）… 単発では13.5秒で成功したが、並列で504、逐次でも504、5日ずつに分割しても504。最後はメタデータを開くだけの最小リクエストすら失敗する状態に

一方で**静的ファイルサーバーは一度も失敗していません**（重い処理をせず、ただファイルを配信するだけのため）。通信量と引き換えに確実さを取る構成にしています。

## 保存先について
Googleドライブは使わず、Colab上の一時ディスクに切り出し済みファイルを貯めてから、**最後に1回だけZIPにしてブラウザ経由でPCにダウンロード**します。

## セッションが切れたら
スクリプトは既にあるファイルをスキップするので、同じセッション内なら③を再実行すれば続きから再開されます。セッションごと切れた場合は①からやり直してください。

## 使い方
**⑤を先に実行**してから、①→②→③→④の順で実行してください。

## ① リポジトリを取得（初回はクローン、2回目以降は最新化のみ）

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/typhoon-dataset-improvements-hn814c'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} https://github.com/awg-yk/typhoon-wind-rainfall {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## ② 必要なライブラリをインストール

In [ ]:
!pip install -q xarray netCDF4

## ③ ダウンロード実行（本体）

全球ファイルを1件ずつ取得 → 日本域に切り出し → 全球ファイル削除、を185回繰り返します。各行に経過時間と残り時間の目安が出ます。

静的ファイルサーバーは同時アクセスに耐えるので、既定で2並列です。速くしたい場合は `--workers 4` などに上げてみてください（失敗が増えるようなら戻す）。地上気圧も欲しい場合は `--include-surface-pressure`、切り出す範囲を変えたい場合は `--north/--south/--west/--east` を追加してください。

最後まで走っても失敗が残った場合は、このセルをもう一度実行すると、失敗した月だけ再取得を試みます（成功済みのファイルはスキップされます）。

In [ ]:
LOCAL_DIR = '/content/jra3q_pressure'

!cd {REPO_DIR} && python scripts/download_jra3q_pressure.py --out-dir "{LOCAL_DIR}"

## ④ ZIPにまとめてPCへダウンロード

In [ ]:
import shutil
import pathlib

from google.colab import files as colab_files

files = list(pathlib.Path(LOCAL_DIR).glob('*.nc'))
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f'{len(files)} files, {total_mb:.1f} MB')

zip_base = '/content/jra3q_pressure_japan'
zip_path = shutil.make_archive(zip_base, 'zip', LOCAL_DIR)
print(f'{zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
colab_files.download(zip_path)

## ⑤ （最初に実行してください）無操作切断を遅らせる

③は時間がかかるため、その間にColabが無操作と判断して切断することがあります。**③より先に**このセルを流しておくと、ブラウザのタブを開いたままにしている間は接続維持の合図を送り続けます（非公式の小技のため過信せず、切れたら①からやり直してください）。

In [ ]:
from IPython.display import Javascript, display

display(Javascript('''
function KeepAlive(){
  console.log("keep-alive ping");
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))

## PCで受け取った後

`jra3q_pressure_japan.zip` を、リポジトリの `data/raw_jra3q/` フォルダなど好きな場所に展開してください。`data/raw_jra3q/` は `.gitignore` 済みなので、そのまま置いてもリポジトリには影響しません。